In [3]:
import duckdb

con = duckdb.connect()

In [5]:
print(type(con))

<class '_duckdb.DuckDBPyConnection'>


In [12]:
import os
os.getcwd()
os.chdir("c:\\Users\\pera.rajahkumar\\OneDrive - Gemeente Waddinxveen\\Bureaublad\\DSBA\\Python\\UKHousingMarket\\data")

In [15]:
con.sql("""
    SELECT *
    FROM read_csv(
        'ppd_data.csv',
        header = false,
        columns = {
            'transaction_id': 'VARCHAR',
            'price': 'BIGINT',
            'date_transfer': 'DATE',
            'postcode': 'VARCHAR',
            'property_type': 'VARCHAR',
            'new_build': 'VARCHAR',
            'duration': 'VARCHAR',
            'paon': 'VARCHAR',
            'saon': 'VARCHAR',
            'street': 'VARCHAR',
            'locality': 'VARCHAR',
            'town': 'VARCHAR',
            'district': 'VARCHAR',
            'county': 'VARCHAR',
            'ppd_category': 'VARCHAR',
            'record_url': 'VARCHAR'
        }
    )
    LIMIT 10
""")

┌──────────────────────────────────────┬────────┬───────────────┬──────────┬───────────────┬───────────┬──────────┬──────────────┬─────────────────────────┬──────────────┬──────────┬─────────┬──────────┬────────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────┐
│            transaction_id            │ price  │ date_transfer │ postcode │ property_type │ new_build │ duration │     paon     │          saon           │    street    │ locality │  town   │ district │     county     │ ppd_category │                                            record_url                                             │
│               varchar                │ int64  │     date      │ varchar  │    varchar    │  varchar  │ varchar  │   varchar    │         varchar         │   varchar    │ varchar  │ varchar │ varchar  │    varchar     │   varchar    │                                              varchar                                        

In [ ]:
#create table transactions

con.sql("""
    CREATE OR REPLACE VIEW transactions AS
    SELECT *
    FROM read_csv(
        'ppd_data.csv',
        header = false,
        columns = {
            'transaction_id': 'VARCHAR',
            'price': 'BIGINT',
            'date_transfer': 'DATE',
            'postcode': 'VARCHAR',
            'property_type': 'VARCHAR',
            'new_build': 'VARCHAR',
            'duration': 'VARCHAR',
            'paon': 'VARCHAR',
            'saon': 'VARCHAR',
            'street': 'VARCHAR',
            'locality': 'VARCHAR',
            'town': 'VARCHAR',
            'district': 'VARCHAR',
            'county': 'VARCHAR',
            'ppd_category': 'VARCHAR',
            'record_url': 'VARCHAR'
        }
    )
""")

In [18]:
con.sql("SELECT COUNT(*) AS total_transactions FROM transactions")

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│             100000 │
└────────────────────┘

In [19]:
con.sql("""
    SELECT
        MIN(date_transfer) AS earliest_sale,
        MAX(date_transfer) AS latest_sale,
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        AVG(price)::INT AS avg_price
    FROM transactions
""")

┌───────────────┬─────────────┬───────────┬───────────┬───────────┐
│ earliest_sale │ latest_sale │ min_price │ max_price │ avg_price │
│     date      │    date     │   int64   │   int64   │   int32   │
├───────────────┼─────────────┼───────────┼───────────┼───────────┤
│ 2021-01-27    │ 2026-07-30  │         1 │  49587470 │    761220 │
└───────────────┴─────────────┴───────────┴───────────┴───────────┘

In [20]:
con.sql("""
    SELECT district, COUNT(*) AS n_sales
    FROM transactions
    GROUP BY district
    ORDER BY n_sales DESC
    LIMIT 15
""")

┌────────────────┬─────────┐
│    district    │ n_sales │
│    varchar     │  int64  │
├────────────────┼─────────┤
│ WANDSWORTH     │    5415 │
│ CROYDON        │    4874 │
│ BROMLEY        │    4654 │
│ BARNET         │    4351 │
│ LAMBETH        │    3977 │
│ EALING         │    3713 │
│ TOWER HAMLETS  │    3692 │
│ HAVERING       │    3599 │
│ LEWISHAM       │    3454 │
│ SOUTHWARK      │    3276 │
│ HILLINGDON     │    3251 │
│ WALTHAM FOREST │    3217 │
│ BEXLEY         │    3171 │
│ GREENWICH      │    3154 │
│ ENFIELD        │    3085 │
├────────────────┴─────────┤
│ 15 rows        2 columns │
└──────────────────────────┘

In [21]:
con.close()

In [25]:
con = duckdb.connect("london_housing.duckdb"
               )

In [26]:
con.sql("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT *
    FROM read_csv(
        'ppd_data.csv',
        header = false,
        columns = {
            'transaction_id': 'VARCHAR',
            'price': 'BIGINT',
            'date_transfer': 'DATE',
            'postcode': 'VARCHAR',
            'property_type': 'VARCHAR',
            'new_build': 'VARCHAR',
            'duration': 'VARCHAR',
            'paon': 'VARCHAR',
            'saon': 'VARCHAR',
            'street': 'VARCHAR',
            'locality': 'VARCHAR',
            'town': 'VARCHAR',
            'district': 'VARCHAR',
            'county': 'VARCHAR',
            'ppd_category': 'VARCHAR',
            'record_url': 'VARCHAR'
        }
    )
""")

In [27]:
con.sql("""
    CREATE OR REPLACE VIEW clean_transactions AS
    SELECT *
    FROM transactions
    WHERE price > 10000
""")

con.sql("SELECT COUNT(*) FROM clean_transactions")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99616 │
└──────────────┘

In [6]:
os.getcwd()

os.chdir("c:\\Users\\pera.rajahkumar\\OneDrive - Gemeente Waddinxveen\\Bureaublad\\DSBA\\Python\\UKHousingMarket")

In [7]:
import duckdb
con = duckdb.connect("data/london_housing.duckdb")
print(con.sql("SELECT DISTINCT district FROM clean_transactions ORDER BY district").df())
con.close()

                  district
0     BARKING AND DAGENHAM
1                   BARNET
2                   BEXLEY
3                    BRENT
4                  BROMLEY
5                   CAMDEN
6           CITY OF LONDON
7      CITY OF WESTMINSTER
8                  CROYDON
9                   EALING
10                 ENFIELD
11               GREENWICH
12                 HACKNEY
13  HAMMERSMITH AND FULHAM
14                HARINGEY
15                  HARROW
16                HAVERING
17              HILLINGDON
18                HOUNSLOW
19               ISLINGTON
20  KENSINGTON AND CHELSEA
21    KINGSTON UPON THAMES
22                 LAMBETH
23                LEWISHAM
24                  MERTON
25                  NEWHAM
26               REDBRIDGE
27    RICHMOND UPON THAMES
28               SOUTHWARK
29                  SUTTON
30           TOWER HAMLETS
31          WALTHAM FOREST
32              WANDSWORTH
